[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 861.4 kB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [15]:
from types import new_class
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.n_head = num_heads
        self.d_k = d_model//num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def attn(self, q, k, v, causal:bool) -> torch.tensor:
      b, _, s, _ = q.shape
      score = q @ k.transpose(-2, -1)/torch.sqrt(torch.tensor(self.d_k))
      if causal:
        mask = torch.triu(torch.ones(s, s, dtype=torch.bool), diagonal=1)
        score = score.masked_fill(mask, -torch.inf)

      attn_weight = torch.softmax(score, dim=-1)
      output = attn_weight @ v
      return output

    def forward(self, x, cache=None):
      b, s, d = x.shape
      if cache is None:
        q = self.wq(x).reshape(b, s, self.n_head, self.d_k).transpose(1, 2)
        k = self.wk(x).reshape(b, s, self.n_head, self.d_k).transpose(1, 2)
        v = self.wv(x).reshape(b, s, self.n_head, self.d_k).transpose(1, 2)

        output = self.attn(q, k, v, True).transpose(1, 2).contiguous().reshape(b, s, d)

        return self.wo(output), (k, v)
      else:
        q = self.wq(x).reshape(b, s, self.n_head, self.d_k).transpose(1, 2)
        k = self.wk(x).reshape(b, s, self.n_head, self.d_k).transpose(1, 2)
        v = self.wv(x).reshape(b, s, self.n_head, self.d_k).transpose(1, 2)

        new_k = torch.cat([cache[0], k], dim=-2)
        new_v = torch.cat([cache[1], v], dim=-2)

        output = self.attn(q, new_k, new_v, False).transpose(1, 2).contiguous().reshape(b, s, d)

        return self.wo(output), (new_k, new_v)

In [16]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (5.4ms)
  ✅ [2/5] Cache structure (3.2ms)
  ✅ [3/5] Decode step appends to cache (2.8ms)
  ✅ [4/5] Incremental decode matches full forward (2.9ms)
  ✅ [5/5] Gradient flow (3.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (17.4ms total)
  Progress saved. Run status() to see your dashboard.

